In [1]:
import os
import re
import logging
import sys
import torch
import torch.nn as nn
from typing import Dict, Tuple
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import subprocess

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)

# Log environment information
logger.info("\n" + "=" * 80)
logger.info("ENVIRONMENT CONFIGURATION")
logger.info("=" * 80)
logger.info(f"Python version: {sys.version}")
logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"CUDA version: {torch.version.cuda}")
    logger.info(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        logger.info(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    logger.info("Running on CPU")

try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], 
                          capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        logger.info(f"NVIDIA GPU: {result.stdout.strip()}")
except Exception:
    pass
logger.info("=" * 80 + "\n")

04/15/2026 00:59:38 - INFO - __main__ - 
04/15/2026 00:59:38 - INFO - __main__ - ENVIRONMENT CONFIGURATION
04/15/2026 00:59:38 - INFO - __main__ - ================================================================================
04/15/2026 00:59:38 - INFO - __main__ - Python version: 3.12.0 (main, Oct  2 2023, 23:53:49) [MSC v.1929 64 bit (AMD64)]
04/15/2026 00:59:38 - INFO - __main__ - PyTorch version: 2.11.0+cpu
04/15/2026 00:59:38 - INFO - __main__ - CUDA available: False
04/15/2026 00:59:38 - INFO - __main__ - Running on CPU
04/15/2026 00:59:38 - INFO - __main__ - ================================================================================



In [ ]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas

!uv pip uninstall transformers tokenizers accelerate -q

!uv pip install "transformers==4.56.0" "protobuf==5.29.3" -q
!uv pip install torch datasets -q
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
# !uv pip install -r requirements.txt
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

In [2]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

def preprocess_code(code_str: str) -> str:
    """Normalize code string."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)
    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()


def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)


def load_datasets(tokenizer, max_length: int = 512) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets."""
    logger.info("Loading datasets from Hugging Face Hub...")
    
    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")
    
    logger.info(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
    
    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
    )
    
    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
    )
    
    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")
    
    return train_dataset, val_dataset

In [3]:
# ============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# ============================================================================

# Note: The Trainer class from transformers handles:
# - Training loop with gradient accumulation
# - Validation every N steps (eval_steps=200)
# - Best model selection based on macro_f1
# - Automatic W&B logging if enabled
# - Checkpointing and model saving

In [5]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(model_name: str, num_labels: int = 2):
    """Load pretrained model and tokenizer from Hugging Face."""
    logger.info(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    logger.info(f"Loading model from: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )
    
    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")
    
    return model, tokenizer

In [ ]:
# ============================================================================
# 4. ORCHESTRATION: Main training pipeline using HF Trainer
# ============================================================================

def compute_metrics_fn(eval_pred):
    """Compute metrics for evaluation."""
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    
    macro_f1 = precision_recall_fscore_support(labels, predictions, average="macro")[2]
    accuracy = accuracy_score(labels, predictions)
    
    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
    }


def train_pipeline(
    model_name: str = "microsoft/codebert-base",
    output_dir: str = "./taskA-codebert-base",
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 2e-5,
    max_length: int = 512,
    num_labels: int = 2,
    use_wandb: bool = False,
):
    """Complete training pipeline using Hugging Face Trainer."""
    from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup file logging to output directory
    log_file = os.path.join(output_dir, "training.log")
    file_handler = logging.FileHandler(log_file, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(name)s - %(message)s"))
    logger.addHandler(file_handler)
    logger.info(f"Logging to file: {log_file}")
    
    # Load model and tokenizer
    model, tokenizer = load_model_and_tokenizer(model_name, num_labels)
    model = model.to(device)
    logger.info(f"Model on device: {device}")
    
    # Log model architecture
    logger.info("\n" + "=" * 80)
    logger.info("MODEL ARCHITECTURE")
    logger.info("=" * 80)
    model_architecture = str(model)
    logger.info(model_architecture)
    logger.info("=" * 80 + "\n")
    
    # Count total and trainable parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Total parameters: {total_params:,}")
    logger.info(f"Trainable parameters: {trainable_params:,}")
    logger.info(f"Non-trainable parameters: {total_params - trainable_params:,}\n")
    
    # Load and prepare datasets
    train_dataset, val_dataset = load_datasets(tokenizer, max_length)
    
    # Setup weights and biases logging if enabled
    if use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "offline"
            logger.info("W&B set to offline mode for notebook stability")
        except Exception as e:
            logger.warning(f"Could not setup W&B: {e}. Continuing without W&B.")
            use_wandb = False
    
    # Calculate training parameters
    steps_per_epoch = max(1, len(train_dataset) // batch_size)
    total_steps = num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        weight_decay=0.01,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        logging_strategy="steps",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to=["wandb"] if use_wandb else [],
        run_name="codebert-base-task-a" if use_wandb else None,
        gradient_checkpointing=True,
        dataloader_pin_memory=True if torch.cuda.is_available() else False,
        seed=42,
    )
    
    # Create trainer
    data_collator = DataCollatorWithPadding(tokenizer)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    
    # Log training configuration
    logger.info("\n" + "=" * 80)
    logger.info("TRAINING CONFIGURATION")
    logger.info("=" * 80)
    logger.info(f"Model: {model_name}")
    logger.info(f"Output directory: {output_dir}")
    logger.info(f"Epochs: {num_epochs}")
    logger.info(f"Batch size: {batch_size}")
    logger.info(f"Learning rate: {learning_rate}")
    logger.info(f"Max sequence length: {max_length}")
    logger.info(f"Warmup steps: {warmup_steps} / {total_steps}")
    logger.info(f"Evaluation strategy: every 200 steps")
    logger.info(f"Gradient checkpointing: True")
    logger.info(f"W&B logging: {use_wandb}")
    logger.info("=" * 80 + "\n")
    
    # Train
    logger.info("Starting training...")
    trainer.train()
    
    logger.info(f"\n🎯 Training complete!")
    logger.info(f"Best model saved to: {os.path.join(output_dir, 'best_model')}")
    
    return trainer, model, tokenizer


# Run training (with use_wandb=False for notebook stability, can be set to True if needed)
trainer, model, tokenizer = train_pipeline(
    model_name="microsoft/codebert-base",
    output_dir="./taskA-codebert-base",
    num_epochs=3,
    batch_size=32,
    learning_rate=2e-5,
    max_length=512,
    use_wandb=False,
)

04/15/2026 01:02:24 - INFO - __main__ - Loading tokenizer from: microsoft/codebert-base
04/15/2026 01:02:25 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
04/15/2026 01:02:25 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/config.json "HTTP/1.1 200 OK"
04/15/2026 01:02:25 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/microsoft/codebert-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
04/15/2026 01:02:25 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/codebert-base/3b0952feddeffad0063f274080e3c23d75e7eb39/tokenizer_config.json "HTTP/1.1 200 OK"
04/15/2026 01:02:25 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/microsoft/codebert-base/tree/main/additional_chat_templates?recursive=false&expan

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
04/15/2026 01:02:27 - INFO - __main__ - Model loaded successfully
04/15/2026 01:02:27 - INFO - __main__ - Model config - num_labels: 2, hidden_size: 768
04/15/2026 01:02:27 - INFO - __main__ - Model on device: cpu
04/15/2026 01:02:27 - INFO - __main__ - Loading datasets from Hugging Face Hub...
04/15/2026 01:02:27 - INFO - httpx - 

Step,Training Loss,Validation Loss


In [ ]:
# ============================================================================
# 5. INFERENCE: Generate predictions on test set
# ============================================================================

def inference_pipeline(
    trainer,
    tokenizer,
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
):
    """Generate predictions on test set using trained model."""
    logger.info("Loading test dataset...")
    test_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test")
    
    # Tokenize test set using same tokenizer
    logger.info("Tokenizing test dataset...")
    test_dataset = test_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in test_dataset.column_names],
        desc="Tokenizing test",
    )
    
    # Generate predictions using the trained model
    logger.info("Generating predictions on test set...")
    predictions = trainer.predict(test_dataset)
    logits = predictions.predictions
    pred_labels = logits.argmax(axis=-1)
    
    # Write to CSV
    logger.info(f"Writing predictions to {output_csv}...")
    with open(output_csv, "w") as f:
        f.write("id,label\n")
        for i, pred in enumerate(pred_labels):
            f.write(f"{i},{pred}\n")
    
    logger.info(f"✅ Predictions saved to {output_csv}")
    logger.info(f"Total predictions: {len(pred_labels)}")
    logger.info(f"Class distribution: {[(pred_labels == i).sum() for i in range(2)]}")
    
    return pred_labels


# Run inference using the trained model
logger.info("\n" + "=" * 80)
logger.info("INFERENCE")
logger.info("=" * 80 + "\n")

predictions = inference_pipeline(
    trainer=trainer,
    tokenizer=tokenizer,
    output_csv="submission.csv",
    batch_size=32,
    max_length=512,
)

In [ ]:
# ============================================================================
# 6. (OPTIONAL) Upload trained model to Hugging Face Hub
# ============================================================================

def upload_to_hub(
    trainer,
    repo_id: str = "dzungpham/taskA-codebert-base",
    commit_message: str = "Upload fine-tuned CodeBERT model"
):
    """Upload the trained model to Hugging Face Hub."""
    try:
        logger.info(f"Uploading model to {repo_id}...")
        trainer.push_to_hub(
            repo_id=repo_id,
            commit_message=commit_message
        )
        logger.info(f"✅ Model uploaded to: https://huggingface.co/{repo_id}")
    except Exception as e:
        logger.warning(f"Could not upload to Hub: {e}")


# Uncomment the line below to upload the model to HF Hub
# upload_to_hub(trainer, repo_id="your-username/taskA-codebert-base")